# 01 · Channel performance: Andrea Bocelli

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eabanoz/bocelli-nostalgia/blob/main/notebooks/01_channel_performance.ipynb)

**Nostalgia on YouTube — a computational social science tutorial**
University of Trento, Department of Sociology and Social Research

---

Produces a complete census of the channel's public uploads, describes its
trajectory, and selects one video for the close comment analysis in
notebook 02.

## Why a full census is affordable

Channel enumeration never touches `search.list`, the expensive endpoint:

| Step | Endpoint | Cost |
|---|---|---|
| Get the uploads playlist ID | `channels.list` | 1 unit |
| List every video | `playlistItems.list` | 1 unit / 50 videos |
| Get statistics | `videos.list` | 1 unit / 50 videos |

The whole census costs under 100 units of the daily 10,000 — versus 100
units for a *single* search call. Framing a question around a known channel
rather than a discovery problem changes the economics completely.

## Before you start

You need a free YouTube Data API key: see
[`docs/api_key_guide.md`](../docs/api_key_guide.md). Store it as a Colab
secret named `YOUTUBE_API_KEY`.

In [1]:
# --- Clone the repo and set the working directory -------------------------
import sys, os, subprocess
from pathlib import Path

REPO = "bocelli-nostalgia"
if "google.colab" in sys.modules:
    if not Path(REPO).exists():
        subprocess.run(["git", "clone", "-q",
                        "https://github.com/eabanoz/bocelli-nostalgia.git"], check=True)
    ROOT = Path(REPO).resolve()
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
for d in ["data/raw", "data/processed", "outputs/figures"]:
    (ROOT / d).mkdir(parents=True, exist_ok=True)

print("Repo root:", ROOT)

Repo root: /content/bocelli-nostalgia


In [2]:
# --- API key from Colab Secrets -------------------------------------------
# Never type a key into a cell: it is saved inside the .ipynb and pushed
# to GitHub. Sidebar > key icon > Add new secret > YOUTUBE_API_KEY.
API_KEY = None
try:
    from google.colab import userdata
    API_KEY = userdata.get("YOUTUBE_API_KEY")
except Exception:
    API_KEY = os.environ.get("YOUTUBE_API_KEY")
if not API_KEY:
    from getpass import getpass
    API_KEY = getpass("YouTube API key (hidden): ")
print("Key loaded:", bool(API_KEY))

Key loaded: True


In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from src.channel_client import ChannelCollector

pd.set_option("display.width", 150); pd.set_option("display.max_colwidth", 60)
plt.rcParams["figure.dpi"] = 110

CHANNEL_ID = "UCb4JB8-ZAeceuR7EPCPOPzg"   # '@andreabocelli'
cc = ChannelCollector(API_KEY)

## 1 · Verify the channel

Confirm the ID resolves to the channel you intended before spending anything
else. IDs copied from third-party sites are wrong more often than you would
expect, and an unnoticed mismatch invalidates everything downstream.

In [ ]:
info = cc.channel_info(CHANNEL_ID)
for k in ["title", "custom_url", "published_at", "country",
          "subscribers", "total_views", "video_count", "uploads_playlist"]:
    v = info[k]
    print(f"{k:18}: {v:,}" if isinstance(v, int) else f"{k:18}: {v}")
print(f"\n{cc.quota_report()}")

assert "bocelli" in (info["title"] + info["custom_url"]).lower(), \
    "Channel ID does not resolve to Bocelli — stop and check."
print("\nChannel verified.")

Two caveats worth a sentence in any write-up:

- `subscriberCount` is **rounded** by the API for large channels.
- Channel-level `viewCount` is a lifetime total including removed videos, so
  it will not equal the sum of current video view counts.

In [ ]:
videos = cc.list_uploads(info["uploads_playlist"])
print(f"\nListed {len(videos):,} videos; channel reports "
      f"{info['video_count']:,}")
print("The playlist is the reliable count. statistics.videoCount is stale "
      "and excludes Shorts inconsistently — report the discrepancy.")
videos.head(3)

In [ ]:
stats = cc.video_stats(videos.video_id.tolist())
df = videos.merge(stats, on="video_id", how="left")
df = df[df.view_count.notna()].copy()
print(f"{len(df):,} videos with statistics | {cc.quota_report()}")
df.to_csv("data/raw/channel_videos_raw.csv", index=False)

## 2 · Feature construction

- **`is_short`** — sub-60-second videos live in a different feed with
  different comment norms; averaging them with full-length uploads produces
  meaningless numbers.
- **`comment_rate`** — comments per view. The key one here: it measures how
  much a video makes people want to *say something*.

In [ ]:
df["published_at"] = pd.to_datetime(df.published_at, utc=True, errors="coerce")
df["year"] = df.published_at.dt.year
df["age_days"] = (pd.Timestamp.now(tz="UTC") - df.published_at).dt.days

# Live/upcoming report duration "P0D" (= 0s); exclude so they are not
# miscounted as Shorts.
df["is_live"] = df.live_status.isin(["live", "upcoming"]) | \
                (df.duration_iso == "P0D")
df["is_short"] = df.duration_sec.between(1, 60) & ~df.is_live
df["duration_min"] = df.duration_sec / 60
df["engagement_rate"] = df.like_count / df.view_count.replace(0, np.nan)
df["comment_rate"] = df.comment_count / df.view_count.replace(0, np.nan)
df["views_per_day"] = df.view_count / df.age_days.clip(lower=1)

print(f"Period       : {df.year.min():.0f}–{df.year.max():.0f}")
print(f"Shorts       : {df.is_short.sum():,}")
print(f"Live/upcoming: {df.is_live.sum():,}")
print(f"Comments off : {df.comments_disabled.sum():,}")
print(f"Median views : {df.view_count.median():,.0f}")

### The distribution is itself a finding

View counts on an artist channel are brutally right-skewed. Plot on a log
scale — a linear axis shows one spike and a flat line.

The skew is substantive: "the channel's audience" is not one population.
People who arrived at a canonical song through a decade of algorithmic
recommendation differ from those who watch routine uploads.

In [ ]:
full = df[~df.is_short & ~df.is_live]

fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))
ax[0].hist(np.log10(full.view_count.clip(lower=1)), bins=40, color="#4C72B0")
ax[0].set_title("View counts (log10)")
ax[1].hist(full.engagement_rate.dropna()*100, bins=40, range=(0, 5),
           color="#55A868"); ax[1].set_title("Likes per 100 views")
ax[2].hist(full.comment_rate.dropna()*1000, bins=40, range=(0, 3),
           color="#C44E52"); ax[2].set_title("Comments per 1,000 views")
plt.tight_layout()
plt.savefig("outputs/figures/distributions.png", bbox_inches="tight")
plt.show()

sv = full.view_count.sort_values(ascending=False)
print("Share of total views held by the top N videos:")
for n in [1, 5, 10, 25, 50]:
    if n <= len(sv):
        print(f"  top {n:>3}: {100*sv.head(n).sum()/sv.sum():5.1f}%")

## 3 · The channel over time

Two different things, and confusing them is the classic error:

- **Uploads per year** describes *production*.
- **Median views by upload year** describes *accumulated reception*, and is
  confounded by age — a 2010 video has had fifteen years to gather views.

Students read the second as "the channel is declining." Usually it just means
recent videos are young.

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(13, 7))
df.groupby("year").size().plot.bar(ax=ax[0, 0], color="#4C72B0")
ax[0, 0].set_title("Uploads per year"); ax[0, 0].set_xlabel("")
df.groupby("year").view_count.median().plot(ax=ax[0, 1], marker="o",
                                            color="#C44E52")
ax[0, 1].set_title("Median views by upload year (age-confounded)")
ax[0, 1].set_yscale("log"); ax[0, 1].set_xlabel("")
df.groupby("year").engagement_rate.median().mul(100).plot(
    ax=ax[1, 0], marker="o", color="#55A868")
ax[1, 0].set_title("Median likes per 100 views"); ax[1, 0].set_xlabel("")
df.groupby("year").views_per_day.median().plot(ax=ax[1, 1], marker="o",
                                               color="#937860")
ax[1, 1].set_title("Median views per DAY since upload (age-adjusted)")
ax[1, 1].set_yscale("log"); ax[1, 1].set_xlabel("")
plt.tight_layout()
plt.savefig("outputs/figures/channel_over_time.png", bbox_inches="tight")
plt.show()

print("Compare the two right-hand panels. Different story? Then you have")
print("just shown why raw cumulative counts mislead.")

### Watch for catalogue back-fill

If one year holds a large share of uploads, the channel probably uploaded an
archive in bulk. On this channel, 192 of 596 videos went up in 2015, and 125
of those carry a performance year in the title ranging from 1995 to 2015.

**`published_at` is the upload date, not the performance date.** For an
archival re-release those are decades apart, and that gap is the analytic
core of notebook 02.

In [ ]:
import re
def perf_year(t):
    m = re.findall(r"\b(19[5-9]\d|20[0-2]\d)\b", str(t))
    return int(m[-1]) if m else np.nan
df["perf_year"] = df.title.map(perf_year)

spike = df.year.value_counts().idxmax()
s = df[df.year == spike]
print(f"Largest upload year: {spike} with {len(s)} videos")
print(f"  of which {s.perf_year.notna().sum()} name a performance year")
if s.perf_year.notna().any():
    print(f"  performance years span "
          f"{s.perf_year.min():.0f}–{s.perf_year.max():.0f}")

## 4 · Selecting the video for close analysis

The obvious move is the most-viewed video. Look at three rankings first —
they will not agree, and the disagreement is informative.

For a nostalgia study **comment count matters more than view count**: we need
text. A video with 50M views and comments disabled is useless here.

In [ ]:
cols = ["video_id", "title", "year", "view_count", "comment_count",
        "comment_rate"]
for label, tab in [
    ("TOP 8 BY VIEWS", full.nlargest(8, "view_count")),
    ("TOP 8 BY COMMENT COUNT", full.nlargest(8, "comment_count")),
    ("TOP 8 BY COMMENT RATE (min 100k views)",
     full[full.view_count > 100_000].nlargest(8, "comment_rate")),
]:
    print("=" * 76); print(label); print("=" * 76)
    display(tab[cols].reset_index(drop=True))

### A caution about automatic selection

A score weighting comment *volume* and comment *rate* will favour recent
videos, because new uploads accumulate comments quickly relative to views.
On this channel that put a 2024 Spanish-language collaboration at the top —
technically the highest scorer, and a poor choice for studying nostalgia.

**Selection criteria are research decisions, not optimisation problems.**
We choose the 1997 performance: the deepest temporal gap, a canonical
nostalgia object, and ample comments.

In [ ]:
TARGET_VIDEO_ID = "TdWEhMOrRpQ"    # Con Te Partirò, Piazza dei Cavalieri 1997

t = df[df.video_id == TARGET_VIDEO_ID]
if len(t):
    r = t.iloc[0]
    print(f"SELECTED: {r.title}")
    print(f"  published : {r.published_at.date()}  (performance: 1997)")
    print(f"  views     : {r.view_count:,.0f}")
    print(f"  comments  : {r.comment_count:,.0f}")
    print(f"  url       : https://www.youtube.com/watch?v={r.video_id}")
    print(f"\nHarvest cost in notebook 02: "
          f"~{min(int(r.comment_count/100)+1, 300):,} units")
else:
    print("Target not found in this census — check the ID.")

In [ ]:
df.to_csv("data/processed/channel_videos.csv", index=False)
pd.DataFrame([info]).to_csv("data/processed/channel_info.csv", index=False)
Path("data/processed/target_video.txt").write_text(TARGET_VIDEO_ID)
print(f"Saved {len(df):,} videos to data/processed/")
print(cc.quota_report())

In [ ]:
# --- Download the outputs --------------------------------------------------
import shutil
shutil.make_archive("/content/bocelli_step01", "zip", ROOT / "data/processed")
try:
    from google.colab import files
    files.download("/content/bocelli_step01.zip")
except Exception:
    print("Not in Colab — files are in data/processed/")

---

## What to report

1. A full channel census cost under 100 units.
2. `statistics.videoCount` (284) disagreed with the uploads playlist (596).
   The playlist is correct.
3. Attention is concentrated in a handful of videos.
4. Cumulative view counts are age-confounded.
5. 2015 was a catalogue back-fill: upload date ≠ performance date.

**Next:** [`02_harvest_and_prepare.ipynb`](02_harvest_and_prepare.ipynb)